Basic Processing

In [2]:
import pandas as pd
sheets = pd.read_excel("./data/Scats Data October 2006.xls", sheet_name=None)

removed_days = sheets["Notes"]["Unnamed: 1"].dropna().to_list()[5:]
removed_days = [int(i) for i in removed_days]
# removed_days

df_info = sheets["Summary Of Data"].loc[2:]
df_info.columns = df_info.iloc[0]
df_info = df_info[1:]
df_info["SCATS Number"] = df_info["SCATS Number"].ffill()

df_info.dropna(axis=1, inplace=True)
df_info = df_info.convert_dtypes().infer_objects()

df_info_filtered = df_info[df_info["Total"].between(7, 31)]
# df_info_filtered


# imagine having good formatting
df = sheets["Data"]
df.columns = pd.Series(df.loc[0])
df = df.loc[1:]

# remove added time
df.loc[:, 'Date'] = pd.to_datetime(df['Date']).dt.date
df = df.convert_dtypes()

# filter good locations
df = df.loc[df["Location"].isin(df_info_filtered["Location"])]

# add location identifier: SCARS Number + VicRoads Internal
df.loc[:, "Identifier"] = df["SCATS Number"].astype(str) + " - " + df["HF VicRoads Internal"].astype(str)
df.columns = df.columns.str.strip()


df


,SCATS Number,Location,CD_MELWAY,NB_LATITUDE,NB_LONGITUDE,HF VicRoads Internal,VR Internal Stat,VR Internal Loc,NB_TYPE_SURVEY,Date,...,V87,V88,V89,V90,V91,V92,V93,V94,V95,Identifier
1,0970,WARRIGAL_RD N of HIGH STREET_RD,060 G10,-37.86703,145.09159,249,182,1,1,2006-10-01,...,97,97,66,81,50,59,47,29,34,0970 - 249
2,0970,WARRIGAL_RD N of HIGH STREET_RD,060 G10,-37.86703,145.09159,249,182,1,1,2006-10-02,...,102,107,114,80,60,62,48,44,26,0970 - 249
3,0970,WARRIGAL_RD N of HIGH STREET_RD,060 G10,-37.86703,145.09159,249,182,1,1,2006-10-03,...,132,114,86,93,90,73,57,29,40,0970 - 249
4,0970,WARRIGAL_RD N of HIGH STREET_RD,060 G10,-37.86703,145.09159,249,182,1,1,2006-10-04,...,113,132,101,113,90,78,66,52,44,0970 - 249
5,0970,WARRIGAL_RD N of HIGH STREET_RD,060 G10,-37.86703,145.09159,249,182,1,1,2006-10-05,...,120,116,113,99,91,61,55,49,36,0970 - 249
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4188,4821,VICTORIA_ST W OF BURNLEY_ST,002HF02,-37.81296,145.0083,6673,1513,7,1,2006-10-27,...,121,127,103,122,124,117,99,108,88,4821 - 6673
4189,4821,VICTORIA_ST W OF BURNLEY_ST,002HF02,-37.81296,145.0083,6673,1513,7,1,2006-10-28,...,93,93,105,105,112,82,97,106,107,4821 - 6673
4190,4821,VICTORIA_ST W OF BURNLEY_ST,002HF02,-37.81296,145.0083,6673,1513,7,1,2006-10-29,...,118,83,76,66,64,77,60,49,45,4821 - 6673
4191,4821,VICTORIA_ST W OF BURNLEY_ST,002HF02,-37.81296,145.0083,6673,1513,7,1,2006-10-30,...,88,89,80,74,48,67,62,50,62,4821 - 6673


In [5]:
import numpy as np
groups = df.groupby(by="Identifier")
numbered_colnames = list(range(96 * 31))
vcols = [f"V{str(i).zfill(2)}" for i in range(96)]
df_processed = pd.DataFrame()

# add latitude and longtitude
lat_dict = {}
long_dict = {}
identifiers = []
locations_dict = {}

for id, group in groups:
    # all latitude and longtitude is the same
    lat_dict[id] = group["NB_LATITUDE"].iloc[0]
    long_dict[id] = group["NB_LONGITUDE"].iloc[0]
    locations_dict[id] = group["Location"].iloc[0].upper()
    
    combined = pd.concat([group[c] for c in vcols], ignore_index=True)
    df_processed[id] = combined
    
    # index
    identifiers.append(id)
    

    
# df_processed.fillna(-1, inplace=True)
df_processed = pd.concat([
    df_processed, pd.DataFrame([lat_dict]), pd.DataFrame([long_dict]), pd.DataFrame([locations_dict])
])

# post processing: row = identifier + lat-long for coordinate + flow

df_processed = df_processed.T.copy()
df_processed.columns = [*df_processed.columns[:-3], "lat", "long", "location"]
df_processed["id"] = [str(i).strip() for i in identifiers]
df_processed = df_processed[df_processed["lat"] != 0]

valid_lengths = df_processed[numbered_colnames].notna().sum(axis=1)
min_valid_len = valid_lengths.min()
print("Shortest complete length:", min_valid_len)

df_trimmed = df_processed.drop(labels=[i for i in numbered_colnames if i > min_valid_len], axis=1)
df_trimmed





Shortest complete length: 2112


,0,1,2,3,4,5,6,7,8,9,...,2107,2108,2109,2110,2111,2112,lat,long,location,id
0970 - 10503,92,31,20,32,42,31,87,108,32,29,...,423,361,365,414,380,422,-37.8676,145.09146,WARRIGAL_RD S OF HIGH STREET_RD,0970 - 10503
0970 - 16116,37,10,8,6,10,9,27,40,9,8,...,138,124,173,176,170,148,-37.86735,145.09195,HIGH STREET_RD E OF WARRIGAL_RD,0970 - 16116
0970 - 249,86,32,26,32,40,36,62,116,23,27,...,354,265,277,358,308,348,-37.86703,145.09159,WARRIGAL_RD N OF HIGH STREET_RD,0970 - 249
0970 - 5887,47,15,10,20,16,42,67,23,16,22,...,351,349,383,318,352,214,-37.86723,145.09103,HIGH STREET_RD W OF WARRIGAL_RD,0970 - 5887
2000 - 10505,110,28,23,30,40,38,72,116,32,35,...,211,252,283,290,287,202,-37.85221,145.09425,WARRIGAL_RD S OF BURWOOD_HWY,2000 - 10505
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4812 - 6302,49,16,15,22,14,28,38,47,11,8,...,44,56,46,60,73,67,-37.82859,145.01644,SWAN_ST NE OF MADDEN_GV,4812 - 6302
4821 - -1,0,1,0,3,2,2,0,0,0,0,...,11,9,12,17,5,9,-37.81285,145.00849,WALMER_ST N OF VICTORIA_ST,4821 - -1
4821 - 14527,25,10,7,12,14,11,38,27,9,11,...,153,94,186,162,180,180,-37.81312,145.00844,BURNLEY_ST S OF VICTORIA_ST,4821 - 14527
4821 - 16911,44,14,14,25,23,20,46,55,13,21,...,167,187,152,180,195,212,-37.81293,145.00865,VICTORIA_ST E OF BURNLEY_ST,4821 - 16911


In [7]:
df_series = df_processed[numbered_colnames]

t = 10 ** 9

# check
# for i in range(len(numbered_colnames)):
#     if df_series[i].hasnans:
#         df_series = df_series.drop(labels=[i], axis=1)
        
# df_series = df_series.astype(np.float32)
df_series.fillna(0, inplace=True)

df_series

/tmp/ipykernel_8623/1988488465.py:11: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_series.fillna(0, inplace=True)
/tmp/ipykernel_8623/1988488465.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_series.fillna(0, inplace=True)


,0,1,2,3,4,5,6,7,8,9,...,2966,2967,2968,2969,2970,2971,2972,2973,2974,2975
0970 - 10503,92,31,20,32,42,31,87,108,32,29,...,39,33,38,58,55,86,119,45,37,66
0970 - 16116,37,10,8,6,10,9,27,40,9,8,...,7,11,11,11,22,32,33,12,15,14
0970 - 249,86,32,26,32,40,36,62,116,23,27,...,28,37,36,32,48,80,98,48,27,33
0970 - 5887,47,15,10,20,16,42,67,23,16,22,...,0,0,0,0,0,0,0,0,0,0
2000 - 10505,110,28,23,30,40,38,72,116,32,35,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4812 - 6302,49,16,15,22,14,28,38,47,11,8,...,0,0,0,0,0,0,0,0,0,0
4821 - -1,0,1,0,3,2,2,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
4821 - 14527,25,10,7,12,14,11,38,27,9,11,...,14,13,32,15,27,42,43,9,12,14
4821 - 16911,44,14,14,25,23,20,46,55,13,21,...,29,19,45,38,36,65,82,28,28,28


In [11]:
data = df_series.to_numpy()

print(data)

windows = []

DAY_LENGTH = 96
WINDOW_LENGTH = DAY_LENGTH * 7 # 1 week
for i in range(WINDOW_LENGTH, len(data.T) + 1):
    windows.append(data.T[i - WINDOW_LENGTH : i])

windows = np.array(windows)

print(windows.shape)

[[92 31 20 ... 45 37 66]
 [37 10  8 ... 12 15 14]
 [86 32 26 ... 48 27 33]
 ...
 [25 10  7 ...  9 12 14]
 [44 14 14 ... 28 28 28]
 [86 34 30 ... 45 62 54]]
(2305, 672, 136)


In [12]:
# ====================================
# WEEKLY LSTM TRAINING SYSTEM
# ====================================
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from keras.models import Sequential
from keras.layers import Input, LSTM, Dense, Dropout
from keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt

# Your windows are shape: (num_windows, 672, 136)
# 672 timesteps = 7 days × 96 intervals/day
# 136 sensors

print(f"Windows shape: {windows.shape}")
print(f"Number of weeks: {windows.shape[0]}")
print(f"Timesteps per week: {windows.shape[1]}")
print(f"Number of sensors: {windows.shape[2]}")

# ====================================
# STEP 1: PREPARE DATA FOR LSTM
# ====================================

# We'll predict NEXT week based on CURRENT week
# Input: week[i] → Output: week[i+1]

X = windows[:-1]  # All weeks except last
y = windows[1:]   # All weeks except first (shifted by 1)

print(f"\nInput shape (X): {X.shape}")   # Should be (num_weeks-1, 672, 136)
print(f"Output shape (y): {y.shape}")    # Should be (num_weeks-1, 672, 136)

# ====================================
# STEP 2: TRAIN/TEST SPLIT
# ====================================

# Split by TIME (not randomly!)
train_ratio = 0.7
val_ratio = 0.15
# test_ratio = 0.15 (implicit)

n_total = len(X)
n_train = int(n_total * train_ratio)
n_val = int(n_total * (train_ratio + val_ratio))

X_train, y_train = X[:n_train], y[:n_train]
X_val, y_val = X[n_train:n_val], y[n_train:n_val]
X_test, y_test = X[n_val:], y[n_val:]

print(f"\n📊 Data Split:")
print(f"  Training: {len(X_train)} weeks")
print(f"  Validation: {len(X_val)} weeks")
print(f"  Test: {len(X_test)} weeks")

# ====================================
# STEP 3: NORMALIZE DATA
# ====================================

# Flatten for normalization
X_train_flat = X_train.reshape(-1, X_train.shape[2])  # (samples*timesteps, sensors)
y_train_flat = y_train.reshape(-1, y_train.shape[2])

# Fit scaler on training data only
scaler_X = StandardScaler().fit(X_train_flat)
scaler_y = StandardScaler().fit(y_train_flat)

# Transform all datasets
def normalize_data(X, y, scaler_X, scaler_y):
    X_norm = scaler_X.transform(X.reshape(-1, X.shape[2])).reshape(X.shape)
    y_norm = scaler_y.transform(y.reshape(-1, y.shape[2])).reshape(y.shape)
    return X_norm, y_norm

X_train_norm, y_train_norm = normalize_data(X_train, y_train, scaler_X, scaler_y)
X_val_norm, y_val_norm = normalize_data(X_val, y_val, scaler_X, scaler_y)
X_test_norm, y_test_norm = normalize_data(X_test, y_test, scaler_X, scaler_y)

print("✅ Data normalized")

# ====================================
# STEP 4: BUILD LSTM MODEL
# ====================================

def build_weekly_lstm(input_shape, output_shape):
    """
    Build LSTM model for weekly traffic prediction.
    
    Architecture:
      Input (week data) 
        → LSTM(256, return_sequences=True)
        → LSTM(128, return_sequences=True)  ← Keep sequence for week-to-week prediction
        → Dropout(0.3)
        → Dense(output_sensors)
    
    Args:
        input_shape: (timesteps, sensors) e.g., (672, 136)
        output_shape: (timesteps, sensors) e.g., (672, 136)
    """
    model = Sequential([
        Input(shape=input_shape),
        
        # LSTM layers that preserve time dimension
        LSTM(256, return_sequences=True, activation='tanh'),
        Dropout(0.3),
        
        LSTM(128, return_sequences=True, activation='tanh'),
        Dropout(0.3),
        
        # Output layer: predict for each timestep
        Dense(output_shape[1], activation='linear')  # output_shape[1] = number of sensors
    ])
    
    model.compile(
        optimizer='adam',
        loss='mse',
        metrics=['mae']
    )
    
    return model


# Build the model
input_shape = (X_train.shape[1], X_train.shape[2])  # (672, 136)
output_shape = (y_train.shape[1], y_train.shape[2])  # (672, 136)

model = build_weekly_lstm(input_shape, output_shape)
model.summary()

# ====================================
# STEP 5: TRAIN THE MODEL
# ====================================

# Callbacks
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

print("\n🚀 Training weekly LSTM model...")
print("="*70)

history = model.fit(
    X_train_norm, y_train_norm,
    validation_data=(X_val_norm, y_val_norm),
    epochs=100,
    batch_size=4,  # Small batch since we have few weeks
    callbacks=[early_stop],
    verbose=1
)

print("✅ Training complete!")

# ====================================
# STEP 6: SAVE MODEL
# ====================================

model.save('model/weekly_lstm.keras')
print("✅ Model saved to model/weekly_lstm.keras")

# Save training history
hist_df = pd.DataFrame(history.history)
hist_df.to_csv('model/weekly_lstm_history.csv', index=False)
print("✅ Training history saved")

# ====================================
# STEP 7: EVALUATE ON TEST SET
# ====================================

print("\n🔮 Evaluating on test set...")

# Predict (normalized)
y_pred_norm = model.predict(X_test_norm)

# Denormalize back to original scale
y_pred = scaler_y.inverse_transform(y_pred_norm.reshape(-1, y_pred_norm.shape[2])).reshape(y_pred_norm.shape)
y_true = scaler_y.inverse_transform(y_test_norm.reshape(-1, y_test_norm.shape[2])).reshape(y_test_norm.shape)

# Calculate metrics
mae = mean_absolute_error(y_true.flatten(), y_pred.flatten())
rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100

print(f"\n{'='*70}")
print(f"📊 TEST SET EVALUATION")
print(f"{'='*70}")
print(f"MAE:  {mae:.2f} vehicles")
print(f"RMSE: {rmse:.2f} vehicles")
print(f"MAPE: {mape:.1f}%")
print(f"{'='*70}\n")

# ====================================
# STEP 8: VISUALIZE PREDICTIONS
# ====================================

# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history.history['loss'], label='Training Loss')
axes[0].plot(history.history['val_loss'], label='Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Training History')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE
axes[1].plot(history.history['mae'], label='Training MAE')
axes[1].plot(history.history['val_mae'], label='Validation MAE')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].set_title('Mean Absolute Error')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('model/weekly_lstm_training.png', dpi=300)
plt.show()
print("✅ Saved: model/weekly_lstm_training.png")


# ====================================
# STEP 9: VISUALIZE SAMPLE PREDICTIONS
# ====================================

# Pick a test week to visualize
test_week_idx = 0
sensor_idx = 0  # First sensor

y_true_week = y_true[test_week_idx, :, sensor_idx]
y_pred_week = y_pred[test_week_idx, :, sensor_idx]

# Create time axis (672 intervals = 7 days)
time_hours = np.arange(672) * 0.25  # Each interval is 15 min = 0.25 hours

fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(time_hours, y_true_week, 'b-', label='Actual', linewidth=2, alpha=0.8)
ax.plot(time_hours, y_pred_week, 'r--', label='Predicted', linewidth=2, alpha=0.8)
ax.fill_between(time_hours, y_true_week, y_pred_week, alpha=0.2, color='gray')

# Mark day boundaries
for day in range(1, 8):
    ax.axvline(x=day*24, color='black', linestyle=':', alpha=0.3, linewidth=1)
    ax.text(day*24, ax.get_ylim()[1]*0.95, f'Day {day}', ha='center', fontsize=9)

sensor_name = df_processed['id'].iloc[sensor_idx]
mae_week = np.mean(np.abs(y_true_week - y_pred_week))

ax.set_xlabel('Hours', fontsize=12, fontweight='bold')
ax.set_ylabel('Traffic Flow (vehicles)', fontsize=12, fontweight='bold')
ax.set_title(f'Weekly Prediction - Sensor: {sensor_name}\nMAE: {mae_week:.2f}', 
            fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('model/weekly_prediction_sample.png', dpi=300)
plt.show()
print("✅ Saved: model/weekly_prediction_sample.png")


# ====================================
# STEP 10: PREDICTION FUNCTION
# ====================================

def predict_next_week(current_week_data):
    """
    Predict traffic for next week given current week's data.
    
    Args:
        current_week_data: ndarray of shape (672, 136) - one week of traffic data
    
    Returns:
        ndarray of shape (672, 136) - predicted next week
    """
    # Normalize
    current_norm = scaler_X.transform(current_week_data.reshape(-1, current_week_data.shape[1])).reshape(1, *current_week_data.shape)
    
    # Predict
    pred_norm = model.predict(current_norm, verbose=0)
    
    # Denormalize
    pred = scaler_y.inverse_transform(pred_norm.reshape(-1, pred_norm.shape[2])).reshape(pred_norm.shape)
    
    return pred[0]


# Example usage
print("\n✨ Prediction function ready!")
print("Use predict_next_week(current_week_data) to predict the next week")

# Test the function
example_week = X_test[0]
predicted_week = predict_next_week(example_week)
print(f"\nExample: Input shape {example_week.shape} → Output shape {predicted_week.shape}")

print("\n🎉 ALL DONE! You now have a weekly traffic prediction LSTM!")

2025-11-13 15:06:22.744669: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-13 15:06:23.572750: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-13 15:06:24.967242: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Windows shape: (2305, 672, 136)
Number of weeks: 2305
Timesteps per week: 672
Number of sensors: 136

Input shape (X): (2304, 672, 136)
Output shape (y): (2304, 672, 136)

📊 Data Split:
  Training: 1612 weeks
  Validation: 346 weeks
  Test: 346 weeks
✅ Data normalized


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 672, 256)       │       402,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 672, 256)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 672, 128)       │       197,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 672, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 672, 136)       │        17,544 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 617,096 (2.35 MB)

 Trainable params: 617,096 (2.35 MB)

 Non-trainable params: 0 (0.00 B)


🚀 Training weekly LSTM model...
Epoch 1/100
403/403 ━━━━━━━━━━━━━━━━━━━━ 162s 398ms/step - loss: 0.2020 - mae: 0.3154 - val_loss: 0.2973 - val_mae: 0.3926
Epoch 2/100
403/403 ━━━━━━━━━━━━━━━━━━━━ 161s 399ms/step - loss: 0.1093 - mae: 0.2349 - val_loss: 0.2197 - val_mae: 0.3402
Epoch 3/100
403/403 ━━━━━━━━━━━━━━━━━━━━ 159s 396ms/step - loss: 0.0892 - mae: 0.2149 - val_loss: 0.2096 - val_mae: 0.3292
Epoch 4/100
403/403 ━━━━━━━━━━━━━━━━━━━━ 160s 397ms/step - loss: 0.0807 - mae: 0.2058 - val_loss: 0.2132 - val_mae: 0.3285
Epoch 5/100
403/403 ━━━━━━━━━━━━━━━━━━━━ 156s 388ms/step - loss: 0.0759 - mae: 0.2007 - val_loss: 0.2091 - val_mae: 0.3236
Epoch 6/100
403/403 ━━━━━━━━━━━━━━━━━━━━ 167s 413ms/step - loss: 0.0721 - mae: 0.1968 - val_loss: 0.1988 - val_mae: 0.3158
Epoch 7/100
403/403 ━━━━━━━━━━━━━━━━━━━━ 519s 1s/step - loss: 0.0693 - mae: 0.1940 - val_loss: 0.1951 - val_mae: 0.3119
Epoch 8/100
185/403 ━━━━━━━━━━━━━━━━━━━━ 3:13 890ms/step - loss: 0.0669 - mae: 0.1909

KeyboardInterrupt: 